In [ ]:
!pip install -q langchain
!pip install -q chromadb
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
from google.colab import userdata

try:
    key = userdata.get("GEMINI_KEY")

    if key:
        print(" Secret found")
        print("Key length:", len(key))
    else:
        print(" Secret is empty")

except Exception as e:
    print("Error:", e)

 Secret found
Key length: 53


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")

print("API key loaded")

API key loaded


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embeddings loaded successfully")

Embeddings loaded successfully


In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

print("Database loaded successfully")

/tmp/ipykernel_2669/1009539281.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Database loaded successfully


In [ ]:
print(vectorstore._collection.count())

0


In [ ]:
import os

print(os.listdir("."))

['.config', 'chroma_db', 'sample_data']


In [ ]:
import os

for root, dirs, files in os.walk("./chroma_db"):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", files)
    print("-" * 50)

ROOT: ./chroma_db
DIRS: []
FILES: ['chroma.sqlite3']
--------------------------------------------------


In [ ]:
os.environ["GEMINI_KEY"] = userdata.get("GEMINI_KEY")
os.environ["GROQ_KEY"] = userdata.get("GROQ_KEY")

In [ ]:
from groq import Groq
client = Groq(
    api_key=os.environ["GROQ_KEY"]
)

In [ ]:
query = "What is Economics?"

docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content[:300])

In [ ]:
context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Use the provided context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Economics is the social science that studies how individuals, businesses, governments, and societies allocate resources to meet their unlimited wants and needs, and how these resources are distributed to create goods and services. It examines the production, distribution, and consumption of goods and services, as well as the interactions among economic agents and the impact of economic activity on the overall well-being of individuals and societies.


In [ ]:
def ask(question):
    docs = vectorstore.similarity_search(question, k=3)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
print(ask("What is scarcity?"))

Scarcity refers to the fundamental economic concept that the needs and wants of individuals are unlimited, but the resources available to satisfy those needs and wants are limited. This means that people have to make choices about how to allocate the limited resources they have in order to meet their needs and wants. Scarcity is the driving force behind economics, as it necessitates the allocation of resources, the creation of economic systems, and the development of methods to manage and distribute resources efficiently. In essence, scarcity is the state of not having enough resources to fulfill all desires or needs, leading to the need for decision-making, prioritization, and resource allocation.
